<a href="https://colab.research.google.com/github/aladdin4220243/aass1122/blob/main/YouTube.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!apt-get install -y p7zip-full >/dev/null 2>&1
!apt -q install ffmpeg >/dev/null 2>&1
!pip -q install -U yt-dlp ipywidgets

import os, subprocess
from google.colab import files
from ipywidgets import Dropdown, interact_manual
from IPython.display import display

URL = input("الرجاء إدخال رابط الفيديو (URL): ")

quality_options = {
    "أفضل جودة (MP4)": "best[ext=mp4]/best",
    "1080p": "best[height<=1080][ext=mp4]",
    "720p": "best[height<=720][ext=mp4]",
    "480p": "best[height<=480][ext=mp4]",
    "360p": "best[height<=360][ext=mp4]",
    "240p": "best[height<=240][ext=mp4]",
    "أقل جودة": "worst[ext=mp4]/worst",
}
target_options = {
    "10% (أقصى ضغط — يناسب فقط الفيديوهات القصيرة)": 0.10,
    "20% من الحجم": 0.20,
    "30% من الحجم": 0.30,
    "50% من الحجم (جودة مقبولة)": 0.50,
}

RESOLUTION = 216        # غيّرها إلى 144 إذا أردت أصغر
MIN_VIDEO_KBPS = 80     # أقل من ذلك = فيديو غير قابل للمشاهدة
KEEP_AUDIO = True       # False = إزالة الصوت نهائياً (توفير إضافي)

quality_dd = Dropdown(options=quality_options, value="best[height<=360][ext=mp4]", description='الجودة:')
target_dd = Dropdown(options=target_options, value=0.20, description='حجم الناتج:')

def download_and_process_video(fmt, pct):
    for f in os.listdir('.'):
        if f.startswith('video.') and not f.endswith(('.7z', '.log')):
            os.remove(f)
    print(f"▶ تحميل: {fmt}")
    os.system(f'yt-dlp -f "{fmt}" -o "video.%(ext)s" "{URL}"')

    video_file = next((f for f in os.listdir('.') if f.startswith('video.') and not f.endswith(('.7z', '.log'))), None)
    if not video_file:
        print("✗ فشل التحميل — يوتيوب يحظر IP كولاب أحياناً، أعد المحاولة")
        return

    src_mb = os.path.getsize(video_file) / 1048576
    dur = float(subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', video_file],
                               capture_output=True, text=True).stdout.strip())

    audio_kbps = 0 if not KEEP_AUDIO else 32
    target_mb = round(src_mb * pct, 1)
    video_kbps = int(target_mb * 8192 / dur - audio_kbps)

    if video_kbps < MIN_VIDEO_KBPS:
        video_kbps = MIN_VIDEO_KBPS
        real_mb = (video_kbps + audio_kbps) * dur / 8192
        print(f"⚠️ الهدف {target_mb:.1f}MB غير واقعي لمدة {dur:.0f}ث — تم رفع البتريت إلى {MIN_VIDEO_KBPS}kbps")
    else:
        real_mb = target_mb
    print(f"الأصلي {src_mb:.1f}MB | المدة {dur:.0f}ث | فيديو {video_kbps}kbps | الناتج المتوقع ~{real_mb:.1f}MB")

    vf = f"scale=-2:{RESOLUTION},fps=24"
    if os.path.exists('video_min.mp4'): os.remove('video_min.mp4')
    os.system(f'ffmpeg -y -i "{video_file}" -vf "{vf}" -c:v libx265 -b:v {video_kbps}k -preset veryfast -pass 1 -an -f null /dev/null 2>/dev/null')
    a_opts = f'-c:a aac -b:a {audio_kbps}k' if KEEP_AUDIO else '-an'
    os.system(f'ffmpeg -y -i "{video_file}" -vf "{vf}" -c:v libx265 -b:v {video_kbps}k -preset veryfast -pass 2 {a_opts} -movflags +faststart video_min.mp4 2>/dev/null')

    os.system('7za a -mx=9 video_min.mp4.7z video_min.mp4 >/dev/null')
    os.system(f'ls -lh "{video_file}" video_min.mp4 video_min.mp4.7z')
    files.download('video_min.mp4.7z')

display(interact_manual(download_and_process_video, fmt=quality_dd, pct=target_dd))